### EMBEDDING AND VECTOR DB

In [1]:
import json
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
import chromadb

In [2]:
CHUNKS_PATH     = "../data/processed/chunks.json"
CHROMA_PATH     = "../data/processed/chroma_db"
COLLECTION_NAME = "elte_ik"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

In [3]:
class EmbeddingPipeline:
    def __init__(self, model_name: str = EMBEDDING_MODEL):
        self.model = SentenceTransformer(model_name)
        print(f"Loaded model: {model_name}")

    def encode(self, texts: list[str]) -> np.ndarray:
        return self.model.encode(texts, show_progress_bar=True)

In [4]:
with open(CHUNKS_PATH, encoding="utf-8") as f:
    chunks = json.load(f)
print(f"Loaded {len(chunks)} chunks")

pipeline = EmbeddingPipeline()
texts = [c["content"] for c in chunks]
embeddings = pipeline.encode(texts)
print(f"Embeddings shape: {embeddings.shape}") 

Loaded 378 chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model: all-MiniLM-L6-v2


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Embeddings shape: (378, 384)


In [5]:
client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

collection.upsert(
    ids=[str(c["metadata"]["chunk_id"]) for c in chunks],
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=[c["metadata"] for c in chunks]
)
print(f"Upserted {collection.count()} documents into '{COLLECTION_NAME}'")

Upserted 378 documents into 'elte_ik'


In [7]:
query = "what is Notification of accommodation?"
query_emb = pipeline.encode([query]).tolist()

results = collection.query(query_embeddings=query_emb, n_results=3)
for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"\n--- Result {i+1} (chunk {meta['chunk_id']}, {meta['file_name']}) ---")
    print(doc[:300])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


--- Result 1 (chunk 57, Enter%20Hungary%20Guide%202023.03..pdf) ---
6. Change of Accommodation
Click on New application → Announcement → Notification of Change of Accommodation

--- Result 2 (chunk 67, Notification%20of%20accommodation.Fill%20in%20guide.pdf) ---
Notification of accommodation 
Fill in guide
ELTE Department of Erasmus+ and International Programmes 2023
visa@elte.hu

--- Result 3 (chunk 58, Enter%20Hungary%20Guide%202023.03..pdf) ---
1. Click on edit and fill out the required information. 
When you are finished, click on save.
2. Click on „Notification of change of 
accommodation”. If you’re finished, click on save.
3. Click on „file attachments” and upload the
required documents.
4. Don’t forget to submit the application!


In [ ]:
import pandas as pd

all_data = collection.get(include=["documents", "metadatas", "embeddings"])

df = pd.DataFrame({
    "chunk_id": [m["chunk_id"] for m in all_data["metadatas"]],
    "file_name": [m["file_name"] for m in all_data["metadatas"]],
    "file_type": [m["file_type"] for m in all_data["metadatas"]],
    "content_preview": [d[:80] + "..." for d in all_data["documents"]],
    "embedding_dim": [len(e) for e in all_data["embeddings"]],
})
print(df.to_string(index=False))

 chunk_id                        file_name file_type                                                                        content_preview  embedding_dim
        0 ELTE Faculty of Informatics.html      html ELTE Faculty of Informatics\nSkip to main content\nELTE Faculty of Informatics\nBSc...            384
        1 ELTE Faculty of Informatics.html      html  23.02.2026.\nBecome an EU Careers Student Ambassador (2026–2027)\nELTE, in coopera...            384
        2 ELTE Faculty of Informatics.html      html  Student Erasmus Applications for the 2026/27 Academic Year.\n03.02.2026.\nStudent ...            384
        3 ELTE Faculty of Informatics.html      html  Educational Materials Building Resilience Against Disinformation\nMore News\nEvent...            384
        4 ELTE Faculty of Informatics.html      html   The Wonders of Machine Perception: Sensors and What Lies Behind Them\nThe Signals...            384
        5            The prerequisites.pdf       pdf  The prerequisite